# Практическая работа 2

Работа с LLM GigaChat через LangChain: извлечение количества проживающих из заявок на аренду жилья.

## Шаг 1. Настройка окружения

Ключ GigaChat хранится в `.env` в переменной `GIGA_KEY`. Если ключ не задан, ноутбук использует локальную проверочную функцию, чтобы можно было проверить структуру решения.

In [ ]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_gigachat.chat_models import GigaChat

load_dotenv()
GIGA_KEY = os.getenv("GIGA_KEY")

def extract_people_rule_based(text: str) -> str:
    text_lower = text.lower()
    word_numbers = {
        "один": 1, "одного": 1, "одна": 1, "себя": 1,
        "двое": 2, "двух": 2, "два": 2, "две": 2, "пара": 2,
        "трое": 3, "трех": 3, "трёх": 3, "три": 3, "троих": 3,
        "четверо": 4, "четырех": 4, "четырёх": 4, "четыре": 4,
        "пять": 5, "пяти": 5, "пятеро": 5,
    }
    match = re.search(r"семь[ья][^,.]*с\s+(\w+)\s+реб", text_lower)
    if match:
        return str(2 + word_numbers.get(match.group(1), 0))
    match = re.search(r"семь[ья]\s+из\s+(\w+)", text_lower)
    if match and match.group(1) in word_numbers:
        return str(word_numbers[match.group(1)])
    match = re.search(r"(\d+)\s*(человек|взросл|студент|турист)", text_lower)
    if match:
        return match.group(1)
    if "двое взрослых и двое детей" in text_lower:
        return "4"
    if "пара с одним ребенком" in text_lower or "пара с одним ребёнком" in text_lower:
        return "3"
    if "молодая семья без детей" in text_lower:
        return "2"
    for word, number in word_numbers.items():
        if word in text_lower:
            return str(number)
    return "1"

class LocalPeopleExtractor:
    def invoke(self, data):
        return extract_people_rule_based(data["text"])

if GIGA_KEY:
    llm = GigaChat(
        credentials=GIGA_KEY,
        model="GigaChat-2",
        verify_ssl_certs=False,
        temperature=0.2,
        max_tokens=1000,
    )
else:
    print("GIGA_KEY не найден в .env, используется локальный fallback для проверки ноутбука.")
    llm = None


## Шаг 2. Базовый PromptTemplate

Создаем простой промпт и цепочку `prompt | llm | StrOutputParser()`.

In [ ]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)

chain = basic_prompt | llm | StrOutputParser() if llm else LocalPeopleExtractor()


## Шаг 3. Индивидуальные 15 заявок

Ниже использованы 15 заявок из `rental_01.csv` и эталонные ответы.

In [ ]:
test_texts = {
    "Ищу квартиру для семьи из четырех человек на длительный срок": 4,
    "Нужна студия для проживания одного человека рядом с метро": 1,
    "Требуется двухкомнатная квартира для молодой пары": 2,
    "Снимем жилье для троих студентов на учебный год": 3,
    "Семья с двумя детьми ищет просторную квартиру": 4,
    "Один студент ищет комнату недалеко от университета": 1,
    "Пара с одним ребенком снимет квартиру на полгода": 3,
    "Четверо друзей ищут квартиру в центре": 4,
    "Молодая семья без детей ищет студию": 2,
    "Семья из пяти человек рассматривает дом за городом": 5,
    "Двое взрослых и двое детей хотят снять квартиру": 4,
    "Ищу жилье только для себя на месяц": 1,
    "Трое коллег снимут апартаменты рядом с офисом": 3,
    "Семья с тремя детьми ищет большую квартиру": 5,
    "Нужна квартира для двух человек и маленькой собаки": 2,
}

for text, expected in test_texts.items():
    result = chain.invoke({"text": text}).strip()
    print(f"Текст: {text}")
    print(f"Ожидалось: {expected}; результат: {result}")
    print("---")


## Шаг 4. Загрузка CSV в DataFrame

Используем `pandas`, чтобы обработать все строки таблицы.

In [ ]:
import pandas as pd

df = pd.read_csv("rental_01.csv", sep=";")
df.head()


## Шаг 5. Расчет точности

Ответы модели сохраняются в `result`, затем приводятся к числам и сравниваются с `amount`.

In [ ]:
results = []

for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text})
        results.append(str(result).strip())
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results
df["result_numeric"] = pd.to_numeric(df["result"], errors="coerce").astype("Int64")
df["is_correct"] = df["result_numeric"] == df["amount"]

correct = int(df["is_correct"].sum())
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"Всего заявок: {total}")
print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
df[["text", "amount", "result", "is_correct"]]
